In [1]:
# ==============================================================================
# 0. INSTALACE A IMPORTOVÁNÍ
# ==============================================================================
!wget -O appartments_train.csv https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_train.csv
!wget -O appartments_test.csv https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_test.csv
!pip install -q catboost xgboost lightgbm optuna scikit-learn pandas numpy scipy

import pandas as pd
import numpy as np
import re
import time
import warnings
import optuna
from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, RobustScaler, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingRegressor, StackingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping
from catboost import CatBoostRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", message="X does not have valid feature names")

print("\n Soubory jsou stažené a uložené v Colabu. Můžeš pokračovat dál!")

--2026-08-05 12:45:37--  https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9033771 (8.6M) [text/plain]
Saving to: ‘appartments_train.csv’

appartments_train.c 100%[===================>]   8.62M  --.-KB/s    in 0.03s   

2026-08-05 12:45:38 (269 MB/s) - ‘appartments_train.csv’ saved [9033771/9033771]

--2026-08-05 12:45:38--  https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/refs/heads/main/homework/lesson_9/appartments_test.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubuserconte

In [2]:
# ==============================================================================
# 1. HLAVNÍ ŘÍDÍCÍ PANEL (MASTER CONFIG)
# ==============================================================================
USE_CLUSTERING = False
ENCODING_METHOD = "target"      # "target" nebo "onehot"

# --- NASTAVENÍ ---
RANDOM_STATE = 75
N_VALIDATION_RUNS = 10
N_TRIALS_OPTUNA = 18
N_FOLDS_OPTUNA = 5
OUTLIER_THRESHOLD = 0.003
threshold = OUTLIER_THRESHOLD
N_CLUSTERS = 50

# --- Velikosti pomocných sad ---
ES_SIZE = 0.15          # podíl foldu vyhrazený na EARLY STOPPING (fáze 1)

# FIXNÍ PARAMETRY (Základ)
FIXED_PARAMS = {
    'n_estimators': 2000,       # strop pro tuning; při dlouhém běhu snížit na 1200
    'early_stopping_rounds': 150,
    'n_jobs': -1,
    'random_state': RANDOM_STATE
}
FULL_ROUNDS = 8000              # strop pro validaci/finál (reálně ukončí early stopping)
FULL_ES = 300

# Sloupce, které nikdy nejdou do modelu
DROP_COLS = ["price", "id", "address", "text", "first_seen", "last_seen",
             "layout", "price_m2", "date_ts"]

In [3]:
# ==============================================================================
# 2. FEATURE ENGINEERING
# ==============================================================================
def remove_outliers(df, threshold):
    """Volat výhradně na trénovací části dat, nikdy na validaci ani testu."""
    df_c = df.copy()

    for col in ['price', 'area', 'floor', 'total_floors']:
        if col in df_c.columns:
            df_c[col] = pd.to_numeric(df_c[col], errors='coerce')

    # Patro vs. Celkem pater
    if 'floor' in df_c.columns and 'total_floors' in df_c.columns:
        mask_bad_floors = df_c['floor'] > df_c['total_floors']
        df_c.loc[mask_bad_floors, 'total_floors'] = df_c.loc[mask_bad_floors, 'floor']

    df_c['price_m2'] = np.where((df_c.get('area', np.nan) > 0), df_c['price'] / df_c['area'], np.nan)

    if 'prague_district' not in df_c.columns:
        df_c['prague_district'] = 'All'

    def filter_group(x):
        low = x.quantile(threshold)
        high = x.quantile(1 - threshold)
        return x.between(low, high) | x.isna()

    mask = df_c.groupby('prague_district')['price_m2'].transform(filter_group)
    return df_c[mask].drop(columns=['price_m2', 'prague_district'], errors='ignore')


def get_prague_district(addr):
    addr = str(addr).lower()
    match = re.search(r'praha\s?-?(\d{1,2})', addr)
    if match: return f"Praha {match.group(1)}"
    return "Unknown"


def enhance_data(df, fit=False, kmeans_model=None, poi_stats=None):
    """
    fit=True  -> spočítá si statistiky (POI výplně, KMeans) z této sady.
    fit=False -> použije statistiky předané zvenčí.
    Vrací (df, kmeans_model, poi_stats).
    """
    df = df.copy()
    if fit or poi_stats is None:
        poi_stats = {} if poi_stats is None else poi_stats

    # --- 1. ČASOVÉ PŘÍZNAKY ---
    df['date_first'] = pd.to_datetime(df['first_seen'], errors='coerce')
    df['date_last'] = pd.to_datetime(df['last_seen'], errors='coerce')
    min_date = pd.Timestamp('2015-01-01')

    df['days_since'] = (df['date_first'] - min_date).dt.days.fillna(0)
    df['days_active'] = (df['date_last'] - df['date_first']).dt.days.fillna(0).clip(lower=0)

    # --- 2. PARSOVÁNÍ DISPOZIC ---
    def get_rooms(val):
        m = re.search(r'(\d+)', str(val))
        return int(m.group(1)) if m else 1
    df['n_rooms'] = df['layout'].apply(get_rooms)

    df['area'] = pd.to_numeric(df['area'], errors='coerce')
    df['avg_room_size'] = df['area'] / df['n_rooms'].replace(0, 1)

    # --- 3. SMART FILLING ---
    if 'elevator' in df.columns:
        df['elevator'] = df['elevator'].astype(str).fillna('').str.lower().apply(
            lambda x: 1 if 'yes' in x or 'true' in x else 0
        )

    for col in ['ownership', 'condition', 'construction']:
        if col in df.columns: df[col] = df[col].fillna('Unknown')

    df['floor'] = pd.to_numeric(df.get('floor', 0), errors='coerce').fillna(0)
    df['total_floors'] = pd.to_numeric(df.get('total_floors', 0), errors='coerce')
    df['total_floors'] = df['total_floors'].fillna(df['floor']).replace(0, 1)

    df['relative_floor'] = df['floor'] / df['total_floors']
    df['is_top_floor'] = (df['floor'] >= df['total_floors']).astype(int)

    # D) POI Vzdálenosti - PENALIZACE
    poi_dists = ['poi_transport_nearest', 'poi_grocery_nearest', 'poi_school_kindergarten_nearest',
                 'poi_doctors_nearest', 'poi_leisure_time_nearest', 'poi_restaurant_nearest']
    for col in poi_dists:
        if col in df.columns:
            val = pd.to_numeric(df[col], errors='coerce')
            key = f"poi_fill_{col}"
            if fit:
                mx = val.max()
                poi_stats[key] = float(mx * 2) if pd.notna(mx) else 5000.0
            df[col] = np.log1p(val.fillna(poi_stats.get(key, 5000.0)))

    # E) POI Počty a Plochy
    cols_to_zero = ['poi_transport_count', 'poi_grocery_count', 'poi_school_kindergarten_count',
                    'poi_doctors_count', 'poi_leisure_time_count', 'poi_restaurant_count',
                    'cellar_area', 'balcony_area', 'garden_area', 'parking']
    for col in cols_to_zero:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # --- 4. NET AREA ---
    df['net_area'] = df['area'] - df['balcony_area'] - df['cellar_area']
    df['net_area'] = df['net_area'].clip(lower=df['area'] * 0.5)

    # --- 5. LOKALITA ---
    def get_district(addr): return addr.split('-')[-1].strip() if isinstance(addr, str) and '-' in addr else "Unknown"
    df['district'] = df['address'].apply(get_district)
    df['prague_district'] = df['address'].apply(get_prague_district)

    df['gps_lat'] = pd.to_numeric(df['gps_lat'], errors='coerce')
    df['gps_lon'] = pd.to_numeric(df['gps_lon'], errors='coerce')
    df['dist_center'] = np.sqrt((df['gps_lat'] - 50.079)**2 + (df['gps_lon'] - 14.430)**2)

    hrad_lat, hrad_lon = 50.090, 14.400
    df['dist_hrad'] = np.sqrt((df['gps_lat'] - hrad_lat)**2 + (df['gps_lon'] - hrad_lon)**2)

    df['proximity_score'] = 1 / (df['dist_center'] + 0.1)

    # --- 6. TEXTOVÉ FEATURES ---
    df['text'] = df['text'].astype(str).fillna('')
    df['text_lower'] = df['text'].str.lower()

    keywords = {
        'txt_metro': ['metro', 'metra', 'metru'],
        'txt_park': ['park', 'stromovk', 'letn', 'riegr', 'vítkov', 'vitkov'],
        'txt_water': ['vltav', 'nábřež', 'výhled na vodu', 'vyhled na vodu'],
        'txt_reconstructed': ['rekonstru', 'po oprav', 'nové jádro', 'zděné jádro', 'zdařil', 'zdaril'],
        'txt_new': ['novostav', 'nový byt', 'projekt', 'kolaudac', 'developersk'],
        'txt_brick': ['cihl', 'činžovní', 'činžovn', 'skelet'],
        'txt_balcony': ['balk', 'lodž', 'lodz', 'teras', 'zahrádk', 'předzahr'],
        'txt_parking': ['parkov', 'garáž', 'garaz', 'stání'],
        'txt_lift': ['výtah', 'vytah'],
        'txt_ac': ['klimatiza', 'klima'],
        'txt_storage': ['komor', 'šatn', 'sklep'],
        'txt_luxury': ['luxus', 'nadstandard', 'design', 'reziden'],
        'txt_view': ['výhled', 'vyhled', 'panoram', 'světlý', 'slunný'],
        'txt_high_ceilings': ['vysoké stropy', 'vysokými stropy'],
        'txt_bad_condition': ['původ', 'umakart', 'před rekonstru', 'k opravě'],
        'txt_low_floor': ['přízemí', 'prizemi', 'suterén'],
        'txt_coop': ['družst', 'druzst', 'družstevní']
    }
    for col, terms in keywords.items():
        pattern = '|'.join(terms)
        df[col] = df['text_lower'].str.contains(pattern, regex=True).astype(int)

    # --- 7. MICRO-SEGMENTACE ---
    df['segment'] = df['prague_district'].astype(str) + '_' + df['construction'].astype(str)

    # --- 8. CLUSTERING (Volitelné) ---
    coords = df[['gps_lat', 'gps_lon']].fillna(0)
    if USE_CLUSTERING and fit:
        kmeans_model = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE)
        df['geo_cluster'] = kmeans_model.fit_predict(coords).astype(str)
    elif USE_CLUSTERING and kmeans_model is not None:
        df['geo_cluster'] = kmeans_model.predict(coords).astype(str)
    else:
        df['geo_cluster'] = "0"

    df = df.drop(columns=['text_lower', 'date_first', 'date_last'], errors='ignore')
    return df, kmeans_model, poi_stats


# ------------------------------------------------------------------------------
# Jediné místo, kde se fituje preprocessing.
# Fituje se vždy jen na trénovací části, ostatní sady se pouze transformují.
# ------------------------------------------------------------------------------
def ylog(df):
    return np.log1p(pd.to_numeric(df["price"], errors="coerce")).values


def make_matrices(fit_df, other_dfs):
    """fit_df = trénovací data (už po remove_outliers). other_dfs = sady k transformaci."""
    X_fit, km, poi_stats = enhance_data(fit_df, fit=True)
    y_fit = ylog(fit_df)
    X_fit = X_fit.drop(columns=DROP_COLS, errors="ignore")

    p = clone(prep)                       # čistý, nefitnutý preprocessor
    M_fit = p.fit_transform(X_fit, y_fit)

    outs = []
    for d in other_dfs:
        X_o, _, _ = enhance_data(d, fit=False, kmeans_model=km, poi_stats=poi_stats)
        X_o = X_o.drop(columns=DROP_COLS, errors="ignore").reindex(columns=X_fit.columns)
        outs.append(p.transform(X_o))
    return p, M_fit, y_fit, outs


def mape_pct(y_log_true, y_log_pred):
    return mean_absolute_percentage_error(np.expm1(y_log_true), np.expm1(y_log_pred)) * 100

In [4]:
# ==============================================================================
# 3. NAČTENÍ DAT + ŠABLONA PREPROCESSORU
# ==============================================================================
try: train_raw = pd.read_csv("appartments_train.csv"); test = pd.read_csv("appartments_test.csv")
except: train_raw = pd.read_csv("apartments_train.csv"); test = pd.read_csv("apartments_test.csv")
print(f" Data načtena: Train {train_raw.shape}")

# Tady se nic nefituje, potřebujeme jen seznam sloupců pro definici pipeline.
X_ref, _, _ = enhance_data(train_raw, fit=True)
X_ref = X_ref.drop(columns=DROP_COLS, errors='ignore')

num_cols = X_ref.select_dtypes(include=[np.number]).columns.tolist()
cat_all = X_ref.select_dtypes(exclude=[np.number]).columns.tolist()

cat_target_candidates = [c for c in ['district', 'prague_district'] if c in cat_all]
if USE_CLUSTERING and 'geo_cluster' in cat_all: cat_target_candidates.append('geo_cluster')
cat_onehot_candidates = [c for c in cat_all if c not in cat_target_candidates]

if ENCODING_METHOD == "target":
    cat_transformers = [
        ("target_enc", TargetEncoder(target_type="continuous", smooth=10.0, cv=5), cat_target_candidates),
        ("onehot_enc", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                                 ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2))]),
         cat_onehot_candidates)
    ]
else:
    cat_all_for_ohe = cat_target_candidates + cat_onehot_candidates
    cat_transformers = [
        ("onehot_enc", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                                 ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2))]),
         cat_all_for_ohe)
    ]

# 'prep' je od teď jen ŠABLONA - fituje se přes clone() uvnitř make_matrices()
prep = ColumnTransformer([
    ("area_fix", SimpleImputer(strategy="constant", fill_value=0), [c for c in num_cols if 'area' in c]),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scaler", RobustScaler()),
                      ("quant", QuantileTransformer(output_distribution='normal', n_quantiles=500))]),
     [c for c in num_cols if 'area' not in c]),
] + cat_transformers)

print(f" Preprocessor (šablona): {ENCODING_METHOD.upper()} | Time Features Active")

 Data načtena: Train (5000, 32)
 Preprocessor (šablona): TARGET | Time Features Active


In [5]:
# ==============================================================================
# 4. OPTUNA
# ==============================================================================
# Každý fold se připraví samostatně:
#   - outliery se mažou jen z trénovací části foldu
#   - preprocessor se fituje jen na trénovací části foldu
#   - early stopping má vlastní sadu, ne tu skórovanou
print("\n Připravuji CV foldy pro Optunu...")
FOLDS = []
kf_opt = KFold(n_splits=N_FOLDS_OPTUNA, shuffle=True, random_state=RANDOM_STATE)
for k, (tr_idx, va_idx) in enumerate(kf_opt.split(train_raw), 1):
    tr_df = train_raw.iloc[tr_idx]
    va_df = train_raw.iloc[va_idx]                      # validační fold zůstává NEDOTČENÝ
    tr_part, es_part = train_test_split(tr_df, test_size=ES_SIZE, random_state=RANDOM_STATE)
    tr_c = remove_outliers(tr_part, OUTLIER_THRESHOLD)
    _, M_tr, y_tr, (M_es, M_va) = make_matrices(tr_c, [es_part, va_df])
    FOLDS.append({'Xtr': M_tr, 'ytr': y_tr,
                  'Xes': M_es, 'yes': ylog(es_part),
                  'Xva': M_va, 'yva': ylog(va_df)})
    print(f"   Fold {k}: train {M_tr.shape} | es {M_es.shape[0]} | val {M_va.shape[0]}")


def objective_xgb(trial):
    params = {
        'n_estimators': FIXED_PARAMS['n_estimators'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 7.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 7.0, log=True),
        'n_jobs': -1, 'random_state': RANDOM_STATE, 'tree_method': 'hist'
    }
    scores = []
    for f in FOLDS:
        model = XGBRegressor(**params, early_stopping_rounds=FIXED_PARAMS['early_stopping_rounds'])
        model.fit(f['Xtr'], f['ytr'], eval_set=[(f['Xes'], f['yes'])], verbose=False)
        scores.append(mape_pct(f['yva'], model.predict(f['Xva'])))
    return float(np.mean(scores))


def objective_lgbm(trial):
    params = {
        'n_estimators': FIXED_PARAMS['n_estimators'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 7.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 7.0, log=True),
        'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1
    }
    scores = []
    for f in FOLDS:
        model = LGBMRegressor(**params)
        model.fit(f['Xtr'], f['ytr'], eval_set=[(f['Xes'], f['yes'])],
                  callbacks=[early_stopping(FIXED_PARAMS['early_stopping_rounds'], verbose=False)])
        scores.append(mape_pct(f['yva'], model.predict(f['Xva'])))
    return float(np.mean(scores))


def objective_cat(trial):
    params = {
        'iterations': FIXED_PARAMS['n_estimators'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 7.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'loss_function': 'MAE', 'random_seed': RANDOM_STATE, 'verbose': 0, 'allow_writing_files': False
    }
    scores = []
    for f in FOLDS:
        model = CatBoostRegressor(**params)
        model.fit(f['Xtr'], f['ytr'], eval_set=(f['Xes'], f['yes']),
                  early_stopping_rounds=FIXED_PARAMS['early_stopping_rounds'], verbose=False)
        scores.append(mape_pct(f['yva'], model.predict(f['Xva'])))
    return float(np.mean(scores))


def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'max_features': trial.suggest_float('max_features', 0.5, 1.0),
        'n_jobs': -1, 'random_state': RANDOM_STATE
    }
    scores = []
    for f in FOLDS:
        model = RandomForestRegressor(**params)
        model.fit(f['Xtr'], f['ytr'])
        scores.append(mape_pct(f['yva'], model.predict(f['Xva'])))
    return float(np.mean(scores))


print(f"\n1. FÁZE: OPTUNA ({N_TRIALS_OPTUNA} trials)...")
# každá studie má vlastní sampler
study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_OPTUNA)
best_xgb_params = study_xgb.best_params
print(f"XGB Best: {study_xgb.best_value:.3f}%")

study_lgbm = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS_OPTUNA)
best_lgbm_params = study_lgbm.best_params
print(f"LGBM Best: {study_lgbm.best_value:.3f}%")

study_cat = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cat.optimize(objective_cat, n_trials=N_TRIALS_OPTUNA)
best_cat_params = study_cat.best_params
print(f"CAT Best: {study_cat.best_value:.3f}%")

study_rf = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objective_rf, n_trials=N_TRIALS_OPTUNA)
best_rf_params = study_rf.best_params
print(f"RF Best: {study_rf.best_value:.3f}%")

# best_*_params drží jen hyperparametry z Optuny.
# Počty iterací a early stopping se doplňují až při konkrétním fitu.


 Připravuji CV foldy pro Optunu...
   Fold 1: train (3378, 107) | es 600 | val 1000
   Fold 2: train (3378, 106) | es 600 | val 1000
   Fold 3: train (3378, 106) | es 600 | val 1000
   Fold 4: train (3378, 106) | es 600 | val 1000
   Fold 5: train (3378, 108) | es 600 | val 1000

1. FÁZE: OPTUNA (18 trials)...
XGB Best: 9.733%
LGBM Best: 9.832%
CAT Best: 9.796%
RF Best: 10.810%


In [6]:
# ==============================================================================
# Rozdělení jednoho runu:
#   train_raw -> 80 % train_d + 20 % val_d
#   uvnitř train_d běží 5-fold CV -> out-of-fold predikce pro všech 4000 řádků
#   z OOF predikcí se počítají váhy ensemblu a meta-model
#   modely, které se měří na val_d, pak trénují na CELÝCH 80 %
#   val_d -> pouze měření
N_FOLDS_BLEND = 5
ES_INNER = 0.12            # podíl foldu vyhrazený na early stopping
ITER_FACTOR_RUN = 1.25     # fold model (~2800 řádků) -> run model (~3980 řádků)
FINAL_ITER_FACTOR = 1.40   # fold model -> finální model na 100 % dat

print(f"\n{'='*85}\n 2. FÁZE: VALIDACE ({N_VALIDATION_RUNS} runs, OOF blending)\n{'='*85}")


def fit_quartet(Xtr, ytr, Xes=None, yes=None, iters=None):
    """Xes zadané -> early stopping. Jinak fixní počet iterací bez ES."""
    bi = {}

    p = dict(best_xgb_params); p.update({'n_jobs': -1, 'random_state': RANDOM_STATE, 'tree_method': 'hist'})
    if Xes is not None:
        xgb = XGBRegressor(**p, n_estimators=FULL_ROUNDS, early_stopping_rounds=FULL_ES)
        xgb.fit(Xtr, ytr, eval_set=[(Xes, yes)], verbose=False)
        bi['xgb'] = getattr(xgb, "best_iteration", None) or FULL_ROUNDS
    else:
        xgb = XGBRegressor(**p, n_estimators=iters['xgb']); xgb.fit(Xtr, ytr, verbose=False)

    p = dict(best_lgbm_params); p.update({'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1})
    if Xes is not None:
        lgbm = LGBMRegressor(**p, n_estimators=FULL_ROUNDS)
        lgbm.fit(Xtr, ytr, eval_set=[(Xes, yes)], callbacks=[early_stopping(FULL_ES, verbose=False)])
        bi['lgbm'] = lgbm.best_iteration_ or FULL_ROUNDS
    else:
        lgbm = LGBMRegressor(**p, n_estimators=iters['lgbm']); lgbm.fit(Xtr, ytr)

    p = dict(best_cat_params); p.update({'loss_function': 'MAE', 'random_seed': RANDOM_STATE,
                                         'verbose': 0, 'allow_writing_files': False})
    if Xes is not None:
        cat = CatBoostRegressor(**p, iterations=FULL_ROUNDS)
        cat.fit(Xtr, ytr, eval_set=(Xes, yes), early_stopping_rounds=FULL_ES, verbose=False)
        bi['cat'] = cat.best_iteration_ or FULL_ROUNDS
    else:
        cat = CatBoostRegressor(**p, iterations=iters['cat']); cat.fit(Xtr, ytr, verbose=False)

    p = dict(best_rf_params); p.update({'n_jobs': -1, 'random_state': RANDOM_STATE})
    rf = RandomForestRegressor(**p); rf.fit(Xtr, ytr)

    return [xgb, lgbm, cat, rf], bi


res = {k: [] for k in ['XGB', 'LGBM', 'CAT', 'RF', 'VOTE_EQ', 'VOTE_OPT', 'STACK']}
optimal_iters = {"xgb": [], "lgbm": [], "cat": []}
collected_weights = []

for run_i in range(1, N_VALIDATION_RUNS + 1):
    seed = RANDOM_STATE + run_i * 7
    train_d, val_d = train_test_split(train_raw, test_size=0.2, random_state=seed)
    y_oof = ylog(train_d)

    # ---------- A) OOF predikce uvnitř train_d ----------
    print(f"Run {run_i}: OOF foldy", end="")
    P_oof = np.zeros((len(train_d), 4))
    fold_iters = {"xgb": [], "lgbm": [], "cat": []}

    kf_bl = KFold(n_splits=N_FOLDS_BLEND, shuffle=True, random_state=seed)
    for tr_idx, va_idx in kf_bl.split(train_d):
        f_tr, f_va = train_d.iloc[tr_idx], train_d.iloc[va_idx]
        f_tr2, f_es = train_test_split(f_tr, test_size=ES_INNER, random_state=seed)
        f_c = remove_outliers(f_tr2, OUTLIER_THRESHOLD)

        _, Xf, yf, (Xes_f, Xva_f) = make_matrices(f_c, [f_es, f_va])
        models_f, bi = fit_quartet(Xf, yf, Xes_f, ylog(f_es))
        P_oof[va_idx] = np.column_stack([m.predict(Xva_f) for m in models_f])
        for k in fold_iters: fold_iters[k].append(bi[k])
        print(".", end="")

    for k in optimal_iters: optimal_iters[k].append(float(np.mean(fold_iters[k])))

    # ---------- B) váhy + meta-model z OOF predikcí ----------
    preds_oof_real = np.expm1(P_oof)
    true_oof = np.expm1(y_oof)

    def mape_loss(weights):
        w = np.clip(weights, 0, None)
        s = w.sum()
        if s <= 0: return 1e9
        return mean_absolute_percentage_error(true_oof, np.average(preds_oof_real, axis=1, weights=w / s))

    r = minimize(mape_loss, [0.25] * 4, method='SLSQP', bounds=[(0, 1)] * 4,
                 constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))
    opt_w = np.clip(r.x, 0, None); opt_w = opt_w / opt_w.sum()
    collected_weights.append(opt_w)

    meta_model = RidgeCV(alphas=[0.1, 1.0, 10.0])
    meta_model.fit(P_oof, y_oof)

    # ---------- C) modely na CELÝCH 80 %, měření na val_d ----------
    print(" | finální trénink", end="")
    train_c = remove_outliers(train_d, OUTLIER_THRESHOLD)
    iters = {k: int(np.mean(v) * ITER_FACTOR_RUN) for k, v in fold_iters.items()}
    _, X_full_d, y_full_d, (X_va_ready,) = make_matrices(train_c, [val_d])
    models, _ = fit_quartet(X_full_d, y_full_d, iters=iters)

    y_val = ylog(val_d)
    P_va = np.column_stack([m.predict(X_va_ready) for m in models])
    print(" Done.")

    for name, j in zip(['XGB', 'LGBM', 'CAT', 'RF'], range(4)):
        res[name].append(mape_pct(y_val, P_va[:, j]))

    preds_va_real = np.expm1(P_va)
    true_va = np.expm1(y_val)
    mape_eq = mean_absolute_percentage_error(true_va, preds_va_real.mean(axis=1)) * 100
    mape_v = mean_absolute_percentage_error(true_va, np.average(preds_va_real, axis=1, weights=opt_w)) * 100
    mape_s = mape_pct(y_val, meta_model.predict(P_va))
    res['VOTE_EQ'].append(mape_eq); res['VOTE_OPT'].append(mape_v); res['STACK'].append(mape_s)

    print(f"    Váhy (z {len(train_d)} OOF predikcí): XGB:{opt_w[0]:.2f}, LGBM:{opt_w[1]:.2f}, "
          f"CAT:{opt_w[2]:.2f}, RF:{opt_w[3]:.2f}")
    print(f"    XGB:{res['XGB'][-1]:.3f}% | LGBM:{res['LGBM'][-1]:.3f}% | CAT:{res['CAT'][-1]:.3f}%")
    print(f"   VOTING (Opt): {mape_v:.3f}% | STACKING: {mape_s:.3f}% | VOTING (rovnoměrný): {mape_eq:.3f}%")

res_voting, res_stacking = res['VOTE_OPT'], res['STACK']


 2. FÁZE: VALIDACE (10 runs, OOF blending)
Run 1: OOF foldy..... | finální trénink Done.
    Váhy (z 4000 OOF predikcí): XGB:0.41, LGBM:0.06, CAT:0.53, RF:0.00
    XGB:9.389% | LGBM:9.568% | CAT:9.689%
   VOTING (Opt): 9.417% | STACKING: 9.497% | VOTING (rovnoměrný): 9.486%
Run 2: OOF foldy..... | finální trénink Done.
    Váhy (z 4000 OOF predikcí): XGB:0.44, LGBM:0.00, CAT:0.56, RF:0.00
    XGB:9.510% | LGBM:9.591% | CAT:9.875%
   VOTING (Opt): 9.586% | STACKING: 9.438% | VOTING (rovnoměrný): 9.720%
Run 3: OOF foldy..... | finální trénink Done.
    Váhy (z 4000 OOF predikcí): XGB:0.46, LGBM:0.07, CAT:0.47, RF:0.00
    XGB:9.130% | LGBM:9.374% | CAT:9.313%
   VOTING (Opt): 9.079% | STACKING: 9.071% | VOTING (rovnoměrný): 9.241%
Run 4: OOF foldy..... | finální trénink Done.
    Váhy (z 4000 OOF predikcí): XGB:0.45, LGBM:0.09, CAT:0.46, RF:0.00
    XGB:9.448% | LGBM:9.564% | CAT:9.741%
   VOTING (Opt): 9.458% | STACKING: 9.478% | VOTING (rovnoměrný): 9.582%
Run 5: OOF foldy..... | finá

In [7]:
# ==============================================================================
# 6. FÁZE 3: FINÁLNÍ TRÉNINK NA 100 % DAT + SUBMISSIONS
# ==============================================================================
print("\n 3. FÁZE: GENERUJI FINÁLNÍ SUBMISSIONS...")

train_full = remove_outliers(train_raw, OUTLIER_THRESHOLD)
prep_final, X_full_ready, y_full, (X_test_ready,) = make_matrices(train_full, [test])

ix = int(np.mean(optimal_iters["xgb"]) * FINAL_ITER_FACTOR)
il = int(np.mean(optimal_iters["lgbm"]) * FINAL_ITER_FACTOR)
ic = int(np.mean(optimal_iters["cat"]) * FINAL_ITER_FACTOR)
print(f"   Trénuji finální modely (XGB:{ix}, LGBM:{il}, CAT:{ic} iters)...")

xgb_fp = dict(best_xgb_params); xgb_fp.update(
    {'n_estimators': ix, 'n_jobs': -1, 'random_state': RANDOM_STATE, 'tree_method': 'hist'})
lgbm_fp = dict(best_lgbm_params); lgbm_fp.update(
    {'n_estimators': il, 'n_jobs': -1, 'random_state': RANDOM_STATE, 'verbose': -1})
cat_fp = dict(best_cat_params); cat_fp.update(
    {'iterations': ic, 'loss_function': 'MAE', 'random_seed': RANDOM_STATE,
     'verbose': 0, 'allow_writing_files': False})
rf_fp = dict(best_rf_params); rf_fp.update({'n_jobs': -1, 'random_state': RANDOM_STATE})

xgb_f = XGBRegressor(**xgb_fp); xgb_f.fit(X_full_ready, y_full)
lgbm_f = LGBMRegressor(**lgbm_fp); lgbm_f.fit(X_full_ready, y_full)
cat_f = CatBoostRegressor(**cat_fp); cat_f.fit(X_full_ready, y_full)
rf_f = RandomForestRegressor(**rf_fp); rf_f.fit(X_full_ready, y_full)

p_x = xgb_f.predict(X_test_ready); p_l = lgbm_f.predict(X_test_ready)
p_c = cat_f.predict(X_test_ready); p_r = rf_f.predict(X_test_ready)

# --- 1. Voting Export (váhy = průměr z blend sad všech runů) ---
final_weights = np.mean(collected_weights, axis=0)
final_weights /= np.sum(final_weights)
print(f"   Finální váhy: XGB:{final_weights[0]:.3f}, LGBM:{final_weights[1]:.3f}, "
      f"CAT:{final_weights[2]:.3f}, RF:{final_weights[3]:.3f}")

preds_matrix_test_real = np.column_stack([np.expm1(p_x), np.expm1(p_l), np.expm1(p_c), np.expm1(p_r)])
final_vote = np.maximum(0, np.average(preds_matrix_test_real, axis=1, weights=final_weights))
pd.DataFrame({"id": test["id"], "price": final_vote.round(0).astype(int)}).to_csv(
    "submission_OPTUNA_VOTING.csv", index=False)

# --- 2. Stacking Export ---
# StackingRegressor si meta-model učí na out-of-fold predikcích.
print("    Trénuji Stacking Regressor...")
ests = [('xgb', XGBRegressor(**xgb_fp)), ('lgbm', LGBMRegressor(**lgbm_fp)),
        ('cat', CatBoostRegressor(**cat_fp)), ('rf', RandomForestRegressor(**rf_fp))]
cv_stack = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
stack = StackingRegressor(estimators=ests, final_estimator=RidgeCV(), n_jobs=-1, cv=cv_stack)
stack.fit(X_full_ready, y_full)
final_stack = np.maximum(0, np.expm1(stack.predict(X_test_ready)))
pd.DataFrame({"id": test["id"], "price": final_stack.round(0).astype(int)}).to_csv(
    "submission_OPTUNA_STACKING.csv", index=False)
print("    Hotovo.")


 3. FÁZE: GENERUJI FINÁLNÍ SUBMISSIONS...
   Trénuji finální modely (XGB:3480, LGBM:2011, CAT:4110 iters)...
   Finální váhy: XGB:0.397, LGBM:0.138, CAT:0.464, RF:0.000
    Trénuji Stacking Regressor...


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


    Hotovo.


In [8]:
# ==============================================================================
# 7. FINÁLNÍ REPORT
# ==============================================================================
print(f"\n{'='*85}\n FINÁLNÍ REPORT \n{'='*85}")

# --- 1. SOUHRNNÉ VÝSLEDKY ---
print("\n## 1. VÝSLEDKY NA HOLD-OUT VALIDACI (%d běhů)\n" % N_VALIDATION_RUNS)
rows = [("XGBoost (Optuna tuning)", 'XGB'), ("LightGBM (Optuna tuning)", 'LGBM'),
        ("CatBoost (Optuna tuning)", 'CAT'), ("Random Forest (Optuna tuning)", 'RF'),
        ("Voting ensemble (rovnoměrné váhy)", 'VOTE_EQ'),
        ("Voting ensemble (optimalizované váhy)", 'VOTE_OPT'),
        ("Stacking ensemble", 'STACK')]
summary = pd.DataFrame([{"Model": lbl,
                         "MAPE": f"{np.mean(res[k]):.2f} %",
                         "Std. Dev.": f"±{np.std(res[k]):.2f}"} for lbl, k in rows])
print(summary.to_markdown(index=False))

# --- 2. VÍTĚZNÉ HYPERPARAMETRY ---
print(f"\n{'='*85}\n OPTUNA DIAGNOSTIKA: VÍTĚZNÉ PARAMETRY\n{'='*85}")
try:
    for nm, st in [("XGBoost", study_xgb), ("LightGBM", study_lgbm),
                   ("CatBoost", study_cat), ("RandomForest", study_rf)]:
        print(f"\n--- {nm} (CV MAPE: {st.best_value:.4f}%) ---")
        print(pd.Series(st.best_params, name=f"{nm} Hodnoty").to_markdown())
except Exception as e:
    print(f" CHYBA: Nepodařilo se vypsat Optuna parametry: {e}")

# --- 3. DETAILNÍ VÝSLEDKY VALIDACE (Run by Run) ---
print("\n## 3. DETAILNÍ VÝSLEDKY VALIDACE (Run by Run)")
val_df = pd.DataFrame([{
    'Run': i + 1,
    'XGB (%)': res['XGB'][i], 'LGBM (%)': res['LGBM'][i], 'CAT (%)': res['CAT'][i], 'RF (%)': res['RF'][i],
    'STACKING (%)': res['STACK'][i], 'VOTING (%)': res['VOTE_OPT'][i],
    'XGB_W': collected_weights[i][0], 'LGBM_W': collected_weights[i][1],
    'CAT_W': collected_weights[i][2], 'RF_W': collected_weights[i][3],
} for i in range(N_VALIDATION_RUNS)])
print(val_df.to_markdown(index=False, floatfmt=".3f"))

# --- 4. FEATURE IMPORTANCE ---
print(f"\n{'='*85}\n DETAILNÍ ANALÝZA: CO ROZHODUJE O CENĚ (XGBoost)\n{'='*85}")
feature_names = []
try:
    for name, transformer, features in prep_final.transformers_:
        if name in ('num', 'area_fix'):
            feature_names.extend(features)
        elif name == 'onehot_enc':
            try: feature_names.extend(transformer.named_steps['enc'].get_feature_names_out(features))
            except Exception: feature_names.extend(features)
        elif name == 'target_enc':
            feature_names.extend(features)

    if len(feature_names) != len(xgb_f.feature_importances_):
        print(f" Pozor: počet jmen ({len(feature_names)}) nesedí s featurami v modelu "
              f"({len(xgb_f.feature_importances_)}). Zobrazuji raw importances.")
        imp_df = pd.DataFrame({'Feature': [f"Feature_{i}" for i in range(len(xgb_f.feature_importances_))],
                               'Importance': xgb_f.feature_importances_})
    else:
        imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': xgb_f.feature_importances_})

    print(imp_df.sort_values(by='Importance', ascending=False).head(40).to_markdown(index=False, floatfmt=".4f"))
except Exception as e:
    print(f" Nepodařilo se vykreslit Feature Importance: {e}")

# --- 5. KONTROLA VÝSTUPU ---
try:
    submission_df = pd.read_csv("submission_OPTUNA_STACKING.csv")
    submission_df['price'] = submission_df['price'].round(0).astype(np.int64)
    print("\n Kontrola finálního formátu (cena v Kč):")
    print(submission_df.head(5).to_string(index=False))
    print(f"\nCelkový počet predikcí: {len(submission_df)}")
except Exception as e:
    print(f" CHYBA PŘI ZOBRAZENÍ VÝSTUPU: {e}")


 FINÁLNÍ REPORT 

## 1. VÝSLEDKY NA HOLD-OUT VALIDACI (10 běhů)

| Model                                 | MAPE    | Std. Dev.   |
|:--------------------------------------|:--------|:------------|
| XGBoost (Optuna tuning)               | 9.51 %  | ±0.47       |
| LightGBM (Optuna tuning)              | 9.64 %  | ±0.50       |
| CatBoost (Optuna tuning)              | 9.72 %  | ±0.52       |
| Random Forest (Optuna tuning)         | 10.75 % | ±0.53       |
| Voting ensemble (rovnoměrné váhy)     | 9.64 %  | ±0.50       |
| Voting ensemble (optimalizované váhy) | 9.48 %  | ±0.49       |
| Stacking ensemble                     | 9.47 %  | ±0.48       |

 OPTUNA DIAGNOSTIKA: VÍTĚZNÉ PARAMETRY

--- XGBoost (CV MAPE: 9.7328%) ---
|                  |   XGBoost Hodnoty |
|:-----------------|------------------:|
| learning_rate    |         0.0105318 |
| max_depth        |         5         |
| subsample        |         0.714164  |
| colsample_bytree |         0.784503  |
| reg_alpha       